# 02 · CAT 控制回路 + 业务状态机 + SHA-256 审计链

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from bootstrap import build_controller, status
from services.config import load_settings, load_policy
print('trunk root:', ROOT)

In [ ]:
from services.rfq_state_machine import RFQStateMachine, IllegalTransition
sm = RFQStateMachine('RFQ-DEMO')
for st in ['INTAKE','STRUCTURING','DFM','QUOTING','VERIFY','REPLY','CRM_MEM','DONE']:
    sm.transition(st, reason='demo')
print('legal path ->', sm.state, '| terminal:', sm.is_terminal)

In [ ]:
sm2 = RFQStateMachine('RFQ-BAD')
try:
    sm2.transition('DONE')  # NEW -> DONE 非法
    print('ERROR: 未拦截非法转移')
except IllegalTransition as e:
    print('非法转移被拦截 ✅:', str(e)[:80])

In [ ]:
from services.audit import AuditChain
ac = AuditChain('RFQ-AUDIT')
ac.log('intake', {'evidence':'EV-1'}); ac.log('dfm', {'valid':True}); ac.log('quote', {'unit':222.8})
print('audit valid=', ac.verify(), '| head=', ac.head[:16], '| events=', len(ac.events))
# 篡改测试
ac.events[1]['payload']['valid'] = False
print('篡改后 audit valid=', ac.verify(), '(应为 False)')